In [3]:
#connect to cozo 
#please be sure that the snbsf01.db is in the same folder as this script  
from pycozo.client import Client
import pandas as pd
from pycozo.client import Client
client = Client('rocksdb', 'snbsf01.db')

In [4]:
# récupération des données depuis Cozo ( hasTag_edges dataframe)
edge_label = pd.DataFrame(client.run('?[id_e, ln] := *edge_label[id_e, ln]'))
edge_df    = pd.DataFrame(client.run('?[id_e, ns, nd] := *edge[id_e, ns, nd]'))
node_label = pd.DataFrame(client.run('?[id_n, ln] := *node_label[id_n, ln]'))

# filtrer hasTag edges
hasTag_edges = edge_label[edge_label['ln'] == 'hasTag'].merge(edge_df, on='id_e')
# joindre labels des ns/nd
hasTag_edges = hasTag_edges.merge(node_label.rename(columns={'id_n':'ns','ln':'ns_label'}), on='ns')
hasTag_edges = hasTag_edges.merge(node_label.rename(columns={'id_n':'nd','ln':'nd_label'}), on='nd')
hasTag_edges = hasTag_edges[(hasTag_edges['ns_label']=='comment') & (hasTag_edges['nd_label']=='tag')]

In [5]:
#Insert Comment nodes via HOGDB

import pandas as pd

from HOGDB.graph.subgraph import Subgraph          # constructeur Subgraph (API HO-GDB)
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase



# my paths 
COMMENT_CSV = "data/comment_0_0.csv"
TAG_CSV = "data/tag_0_0.csv"
HAS_TAG_CSV = "data/comment_hasTag_tag_0_0.csv"

NODE_BATCH = 5000



# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)


# here  just A CACHE

comment_cache = {}   # dict key : id , val : Node
tag_cache = {}       # key : id , val Node
edge_cache = {}      # (c_id, t_id) , Edge

# here  buffers

# here  buffers
comment_buffer = []
tag_buffer = []      
edge_buffer = []   
print("== Insert Comment nodes ==")

comment_df = pd.read_csv(
    COMMENT_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
comment_buffer = []

for _, row in comment_df.iterrows():
    node = Node(
        labels=[Label("Comment")],   # _node sera ajouté par HO-GDB
        properties=[
            Property("id", int, int(row["id"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("length", int, int(row["length"]))
        ]
    )

    comment_cache[int(row["id"])] = node
    comment_buffer.append(node)

    # batch
    if len(comment_buffer) == NODE_BATCH:
        for n in comment_buffer:
            gs.add_node(n)

                     
        count += NODE_BATCH
        print(f"  committed {count} Comment nodes")
        comment_buffer.clear()

# dernier batch
if comment_buffer:
    for n in comment_buffer:
        gs.add_node(n)
    
    count += len(comment_buffer)

print(f"Total Comment nodes inserted: {count} ")


== Insert Comment nodes ==


In [6]:
#Insert Tag nodes via HOGDB
import pandas as pd

from HOGDB.graph.subgraph import Subgraph          # constructeur Subgraph (API HO-GDB)
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase



# my paths
COMMENT_CSV = "data/comment_0_0.csv"
TAG_CSV = "data/tag_0_0.csv"
HAS_TAG_CSV = "data/comment_hasTag_tag_0_0.csv"

NODE_BATCH = 5000



# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)


# here  A CACHE


tag_cache = {}       # key : id , val Node
   

# here  buffers

# here  buffers

tag_buffer = []      

print(" Insert Tag nodes  : ")

tag_df = pd.read_csv(
    TAG_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
tag_buffer = []

for _, row in tag_df.iterrows():
    node = Node(
       labels=[Label("Tag")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, row["name"]),
            Property("url", str, row["url"])
        ]
    )

    tag_cache[int(row["id"])] = node
    tag_buffer.append(node)

    # batch
    if len(tag_buffer) == NODE_BATCH:
        for n in tag_buffer:
            gs.add_node(n)

                     
        count += NODE_BATCH
        print(f"  committed {count} Comment nodes")
        tag_buffer.clear()

# dernier batch
if tag_buffer:
    for n in tag_buffer:
        gs.add_node(n)
    
    count += len(tag_buffer)

print(f"== Total tag nodes inserted: {count} ==")


== Insert Tag nodes ==


In [7]:
#Insert HAS_TAG edges via HOGDB

print("== Insert HAS_TAG edges ==")

edge_df = pd.read_csv(
    HAS_TAG_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

EDGE_BATCH = 1000
count = 0
edge_buffer = []

for _, row in edge_df.iterrows():
    c_id = int(row["Comment.id"])
    t_id = int(row["Tag.id"])

    # pour la sécurité : nodes doivent exister 
    if c_id not in comment_cache or t_id not in tag_cache:
        continue

    e = Edge(
        comment_cache[c_id],
        tag_cache[t_id],
        Label("HAS_TAG"),
        []
    )

    edge_buffer.append(e)

    if len(edge_buffer) == EDGE_BATCH:
        for edge in edge_buffer:
            gs.add_edge(edge)

        
        count += EDGE_BATCH
        print(f"  committed {count} HAS_TAG edges")
        edge_buffer.clear()

# dernier batch
if edge_buffer:
    for edge in edge_buffer:
        gs.add_edge(edge)

    
    count += len(edge_buffer)

print(f"== Total HAS_TAG edges inserted: {count} ==")


== Insert HAS_TAG edges ==


In [9]:
#get edge via HOGDB

import pandas as pd

from HOGDB.graph.subgraph import Subgraph          
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase

# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)
                         
 
for edge in edge_buffer:
    e= gs.get_edge(edge)
    print(type(e))


AssertionError: 

In [10]:
#get edge via HOGDB v2


c = Node(
    labels=[Label("Comment")],
    properties=[Property("id", int, 1030792151054)]
)

t = Node(
    labels=[Label("Tag")],
    properties=[Property("id", int, 2962)]
)

pattern = Edge(c, t, Label("HAS_TAG"),[])

e = gs.get_edge(pattern)

if e is None:
    print("edge not found")
else:
    print("edge found:", type(e))


AssertionError: 

In [ ]:
#Insert taggedComment subgraphs via HOGDB/

SUBGRAPH_BATCH = 1000



#INSERT taggedComment Subgraphs

print("Insert taggedComment subgraphs ")

buffer = []
count = 0

for (c_id, t_id), e in edge_cache.items():
    sg = Subgraph(
        subgraph_nodes=[
            comment_cache[c_id],
            tag_cache[t_id]
        ],
        subgraph_edges=[e],
        labels=[Label("taggedComment")],
        properties=[
            Property("name", str, f"tc_{c_id}_{t_id}")
        ]
    )

    buffer.append(sg)

    if len(buffer) == SUBGRAPH_BATCH:
        for s in buffer:
            gs.add_subgraph(s)
        count += len(buffer)
        buffer.clear()
        print(f"  inserted {count} subgraphs")

# flush last batch
for s in buffer:
    gs.add_subgraph(s)

print(f"== DONE: {count + len(buffer)} subgraphs inserted ==")


gs.close_connection()
